In [1]:
!nvidia-smi

%cd /content
!git clone https://github.com/nuhaminae/The-Conversion-Engine-Ground-Truth
%cd /content/The-Conversion-Engine-Ground-Truth

Sat May  2 15:20:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
import os, re

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install python-dotenv pyyaml
!pip install scikit-learn matplotlib pyyaml peft

In [3]:
import os
from pathlib import Path
from getpass import getpass

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_DISABLED"] = "true"

REPO = Path("/content/The-Conversion-Engine-Ground-Truth")
assert REPO.exists(), f"Repo not found: {REPO}"
os.chdir(REPO)
print("cwd:", os.getcwd())

hf_token = getpass("HF token, or press Enter to skip: ").strip()
os.environ["HF_TOKEN"] = hf_token
os.environ["WANDB_API_KEY"] = ""

!nvidia-smi

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Eval imports OK")

cwd: /content/The-Conversion-Engine-Ground-Truth
HF token, or press Enter to skip: ··········
Sat May  2 15:21:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |   

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Eval imports OK


In [4]:
import json
from pathlib import Path

required_files = [
    Path("configs/eval_config.yaml"),
    Path("configs/training_config.yaml"),
    Path("src/evaluation/eval_judge.py"),
    Path("models/judge/adapter_model.safetensors"),
    Path("models/judge/adapter_config.json"),
    Path("models/judge/tokenizer.json"),
    Path("tenacious_bench/dpo/held_out_dpo.jsonl"),
]

for path in required_files:
    assert path.exists(), f"Missing: {path}"

print("Required evaluation files exist.")

!ls -lah models/judge
!ls -lah tenacious_bench/dpo

heldout_file = Path("tenacious_bench/dpo/held_out_dpo.jsonl")
count = 0
with heldout_file.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        assert "prompt" in row and row["prompt"].strip()
        assert "chosen" in row and row["chosen"].strip()
        assert "rejected" in row and row["rejected"].strip()
        count += 1

print(heldout_file, count, "valid held-out DPO pairs")

!python -m py_compile src/evaluation/eval_judge.py
print("eval_judge.py syntax OK")

Required evaluation files exist.
total 39M
drwxr-xr-x 2 root root 4.0K May  2 15:20 .
drwxr-xr-x 4 root root 4.0K May  2 15:20 ..
-rw-r--r-- 1 root root 1.3K May  2 15:20 adapter_config.json
-rw-r--r-- 1 root root  22M May  2 15:20 adapter_model.safetensors
-rw-r--r-- 1 root root 3.8K May  2 15:20 chat_template.jinja
-rw-r--r-- 1 root root  454 May  2 15:20 special_tokens_map.json
-rw-r--r-- 1 root root  50K May  2 15:20 tokenizer_config.json
-rw-r--r-- 1 root root  17M May  2 15:20 tokenizer.json
total 256K
drwxr-xr-x 2 root root 4.0K May  2 15:20 .
drwxr-xr-x 6 root root 4.0K May  2 15:20 ..
-rw-r--r-- 1 root root  69K May  2 15:20 dev_dpo.jsonl
-rw-r--r-- 1 root root 1.6K May  2 15:20 dpo_report.json
-rw-r--r-- 1 root root  46K May  2 15:20 held_out_dpo.jsonl
-rw-r--r-- 1 root root 121K May  2 15:20 train_dpo.jsonl
tenacious_bench/dpo/held_out_dpo.jsonl 26 valid held-out DPO pairs
eval_judge.py syntax OK


In [5]:
print("Training reports")
!find reports/training -maxdepth 2 -type f -print || true

print("\nJudge model files")
!find models/judge -maxdepth 2 -type f -print || true

print("\nTraining summary")
!cat reports/training/training_summary.json || true

Training reports
reports/training/dataset_summary.json
reports/training/training_summary.json
reports/training/training_config_used.yaml
reports/training/training_run.log

Judge model files
models/judge/chat_template.jinja
models/judge/adapter_model.safetensors
models/judge/adapter_config.json
models/judge/special_tokens_map.json
models/judge/tokenizer_config.json
models/judge/tokenizer.json

Training summary
{
  "status": "complete",
  "base_model": "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
  "model_dir": "models/judge",
  "checkpoint_dir": "models/checkpoints",
  "report_dir": "reports/training",
  "train_pairs": 65,
  "validation_pairs": 39,
  "train_metrics": {
    "train_runtime": 46.6419,
    "train_samples_per_second": 1.394,
    "train_steps_per_second": 0.364,
    "total_flos": 0.0,
    "train_loss": 0.6785038884948281,
    "epoch": 1.0
  },
  "eval_metrics": {
    "eval_loss": 0.6711041331291199,
    "eval_runtime": 6.2784,
    "eval_samples_per_second": 6.212,
    "eval_step

In [6]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["WANDB_DISABLED"] = "true"

!mkdir -p reports

!PYTHONUNBUFFERED=1 python src/evaluation/eval_judge.py \
  --config configs/eval_config.yaml \
  2>&1 | tee reports/eval_fine_tuned_judge.log

print("\nEvaluation outputs:")
!find reports -maxdepth 2 -type f \( -name "*judge*" -o -name "*metrics*" -o -name "*scores*" -o -name "*.png" \) -print || true

print("\nFine-tuned judge metrics:")
!cat reports/fine_tuned_judge_metrics.json || true

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loaded 26 held-out preference pairs from tenacious_bench/dpo/held_out_dpo.jsonl
Loading reference base model: unsloth/Llama-3.2-1B-Instruct-bnb-4bit
`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Loading policy base model: unsloth/Llama-3.2-1B-Instruct-bnb-4bit
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in th

In [7]:
!zip -r week11_eval_outputs.zip \
  reports/fine_tuned_judge_metrics.json \
  reports/fine_tuned_judge_pair_scores.jsonl \
  reports/fine_tuned_judge_confusion_matrix.png \
  reports/eval_fine_tuned_judge.log \
  configs/eval_config.yaml

from google.colab import files
files.download("week11_eval_outputs.zip")

  adding: reports/fine_tuned_judge_metrics.json (deflated 51%)
  adding: reports/fine_tuned_judge_pair_scores.jsonl (deflated 84%)
  adding: reports/fine_tuned_judge_confusion_matrix.png (deflated 15%)
  adding: reports/eval_fine_tuned_judge.log (deflated 62%)
  adding: configs/eval_config.yaml (deflated 57%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>